# Ablation: Data Simulation Method

This notebook compares models trained on different simulation styles. The model architecture, input length, target, and validation pool are controlled.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

REPO = Path.cwd()
if REPO.name == "notebook":
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.util import (
    CAMERA_MAX,
    CANONICAL_PERIOD,
    COLORS,
    EVAL_STEPS,
    FREQ_PERIODS_STEPS,
    FREQ_SWEEP_EVAL_STEPS,
    STEP_MIN,
    experiment_runtime,
    load_eval_pool,
    make_cosine_target,
    plot_error_time,
    plot_single_rollout_trajectory,
    plot_trajectory_group,
    rising_mask,
    run_ode_mpc,
    show_figure,
)


## Shared Evaluation Helpers

Common rollout, trajectory, and comparison plotting helpers are imported from `src.util`. The notebook cells below keep only the study-specific experiment choices and analysis logic.


## Error Over Rollout Time

In [2]:
plot_error_time([
    ("S1 ODE", COLORS[0], "Style1-Past36-LSTM"),
    ("S2 ODE+noise", COLORS[1], "Style2-Past36-LSTM"),
    ("S3 Stoch+noise", COLORS[2], "Style3-Past36-LSTM"),
    ("S4 Stoch+varE", COLORS[3], "Style4-Past36-LSTM"),
], "Ablation: training simulation style error over time", save_name="ablation-DataSimulationMethod-error")


saved figure: [asset/ablation-DataSimulationMethod-error.png](../asset/ablation-DataSimulationMethod-error.png)

## Train/Valid Style Matrix

In [3]:

def compute_mpc_nmae(experiment_name, eval_style=4, period_steps=CANONICAL_PERIOD, eval_steps=EVAL_STEPS):
    cfg, model, ode, horizon, past_steps, camera_max = experiment_runtime(experiment_name)
    desired = make_cosine_target(period_steps, eval_steps, camera_max)
    cell_nmaes = []
    for record in load_eval_pool(style=eval_style):
        ode_response, _ = run_ode_mpc(
            model,
            record.observed[:past_steps],
            record.stimulus[:past_steps],
            desired,
            record.cell_E,
            horizon,
            ode,
            camera_max,
            context_steps=36,
        )
        cell_nmaes.append(float(np.mean(np.abs(ode_response - desired)) / camera_max))
    return {"nmae_per_cell": cell_nmaes, "nmae_mean": float(np.mean(cell_nmaes)), "nmae_std": float(np.std(cell_nmaes))}

def compute_cross_eval_matrix():
    style_experiments = {
        1: "Style1-Past36-LSTM",
        2: "Style2-Past36-LSTM",
        3: "Style3-Past36-LSTM",
        4: "Style4-Past36-LSTM",
    }
    matrix = {}
    for train_style, experiment_name in style_experiments.items():
        row = {}
        for eval_style in [1, 2, 3, 4]:
            row[f"eval_s{eval_style}"] = compute_mpc_nmae(experiment_name, eval_style=eval_style)["nmae_mean"]
        matrix[f"train_s{train_style}"] = row
    return matrix


def plot_style_matrix(title, save_name=None):
    matrix_data = compute_cross_eval_matrix()
    train_keys = ["train_s1", "train_s2", "train_s3", "train_s4"]
    eval_keys = ["eval_s1", "eval_s2", "eval_s3", "eval_s4"]
    labels = ["S1\n(ODE)", "S2\n(ODE+noise)", "S3\n(Stoch+noise)", "S4\n(Stoch+varE)"]
    matrix = np.array([[matrix_data[train][eval_key] for eval_key in eval_keys] for train in train_keys])

    fig, ax = plt.subplots(figsize=(7, 5.5))
    image = ax.imshow(matrix, cmap="RdYlGn_r", vmin=matrix.min() - 0.005, vmax=matrix.max() + 0.005)
    plt.colorbar(image, ax=ax, label="nMAE")
    ax.set_xticks(range(4)); ax.set_xticklabels(labels, fontsize=9)
    ax.set_yticks(range(4)); ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel("Validation style")
    ax.set_ylabel("Training style")
    ax.set_title(title, fontweight="bold")
    for row in range(4):
        for column in range(4):
            ax.text(
                column,
                row,
                f"{matrix[row, column]:.4f}",
                ha="center",
                va="center",
                fontsize=10,
                fontweight="bold",
                color="white" if matrix[row, column] > matrix.mean() else "black",
            )
    plt.tight_layout()
    show_figure(title, save_name)

plot_style_matrix("Ablation: train/valid simulation style matrix", save_name="ablation-DataSimulationMethod-matrix")


saved figure: [asset/ablation-DataSimulationMethod-matrix.png](../asset/ablation-DataSimulationMethod-matrix.png)

## Rollout Trajectory

In [4]:
plot_trajectory_group([
    ("S1 ODE", COLORS[0], "Style1-Past36-LSTM"),
    ("S2 ODE+noise", COLORS[1], "Style2-Past36-LSTM"),
    ("S3 Stoch+noise", COLORS[2], "Style3-Past36-LSTM"),
    ("S4 Stoch+varE", COLORS[3], "Style4-Past36-LSTM"),
], "Ablation: training simulation style trajectory", save_name="ablation-DataSimulationMethod-trajectory")


saved figure: [asset/ablation-DataSimulationMethod-trajectory.png](../asset/ablation-DataSimulationMethod-trajectory.png)